[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C17_Classical_NLP_Course/03_hmm/03_hmm.ipynb)

# 03 · 隐马尔可夫模型 HMM（纯 numpy 从零）

目标：**不调 hmmlearn**，在**对数空间**从零实现 HMM 的 **前向算法**、**Viterbi 解码**、**后向算法**、**监督参数估计**，并**对拍暴力枚举**所有隐状态路径验证正确，最后做词性标注。

路线：定义 $\pi,A,B$ → 联合概率 → 暴力枚举 $P(O)$ → **对数空间前向(对拍暴力)** → **Viterbi(对拍暴力)** → 后向 + 状态后验 → 监督估计 → ✏️ 练习 → 📖 答案 → 🧪 真实 POS 胶囊。

> 心智模型：**到时刻 t、停在状态 i 的所有路径可汇总成一个量**。前向求和(评估)、Viterbi 取最大(解码)，共用一个 DP 骨架。

## 1 · 定义一个 HMM 并算联合概率

玩具天气 HMM：隐状态 {Rainy, Sunny}，观测 {walk, shop, clean}。联合概率 $P(O,S) = \pi_{s_1}B_{s_1,o_1}\prod_t A_{s_{t-1},s_t}B_{s_t,o_t}$。

In [ ]:
import numpy as np
import itertools
rng = np.random.default_rng(0)

states = ['Rainy', 'Sunny']
obs_vocab = ['walk', 'shop', 'clean']
S, Vo = len(states), len(obs_vocab)
si = {s: i for i, s in enumerate(states)}
oi = {o: i for i, o in enumerate(obs_vocab)}

pi = np.array([0.6, 0.4])                       # 初始分布
A = np.array([[0.7, 0.3],                       # 转移: Rainy->{R,S}
              [0.4, 0.6]])                       #       Sunny->{R,S}
B = np.array([[0.1, 0.4, 0.5],                  # 发射: Rainy->{walk,shop,clean}
              [0.6, 0.3, 0.1]])                  #       Sunny->{...}
assert np.allclose(A.sum(1), 1) and np.allclose(B.sum(1), 1) and abs(pi.sum()-1)<1e-9

def joint_prob(obs, hidden):
    o = [oi[x] for x in obs]; s = [si[x] for x in hidden]
    p = pi[s[0]] * B[s[0], o[0]]
    for t in range(1, len(o)):
        p *= A[s[t-1], s[t]] * B[s[t], o[t]]
    return p

obs = ['walk', 'shop', 'clean']
pj = joint_prob(obs, ['Sunny', 'Rainy', 'Rainy'])
print('P(O, S=[Sunny,Rainy,Rainy]) =', round(pj, 6))
assert pj > 0
print('✅ HMM 定义完成，联合概率可算')

## 2 · 暴力枚举 P(O)（我们的 ground truth）

$P(O) = \sum_S P(O,S)$ 对所有 $S^T$ 条隐状态路径求和。小规模下可枚举——这就是验证前向算法的**绝对参考**。

In [ ]:
def brute_force_likelihood(obs):
    T = len(obs)
    total = 0.0
    for hidden in itertools.product(states, repeat=T):
        total += joint_prob(obs, hidden)
    return total

p_obs = brute_force_likelihood(obs)
print(f'P(O) 暴力枚举 {S**len(obs)} 条路径 = {p_obs:.6f}')
assert 0 < p_obs < 1
print('✅ 暴力 P(O) 就绪 —— 前向算法必须复现这个数')

## 3 · 对数空间前向算法（对拍暴力）

$\alpha_t(j) = (\sum_i \alpha_{t-1}(i)A_{ij})B_{j,o_t}$，$P(O)=\sum_i\alpha_T(i)$。
在**对数空间**做：连乘变连加，求和用 **log-sum-exp**（$\log\sum e^{x_i}=m+\log\sum e^{x_i-m}$）防下溢。

In [ ]:
def logsumexp(x, axis=None):
    x = np.asarray(x, dtype=float)
    if axis is None:
        m = np.max(x)
        return float(m + np.log(np.sum(np.exp(x - m))))
    m = np.max(x, axis=axis, keepdims=True)
    out = m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True))
    return np.squeeze(out, axis=axis)

def forward_log(obs, pi, A, B):
    o = [oi[x] for x in obs]; T = len(o)
    logpi, logA, logB = np.log(pi), np.log(A), np.log(B)
    log_alpha = np.zeros((T, S))
    log_alpha[0] = logpi + logB[:, o[0]]
    for t in range(1, T):
        for j in range(S):
            # log Σ_i alpha_{t-1}(i) A_ij  =  logsumexp_i(log_alpha[t-1] + logA[:,j])
            log_alpha[t, j] = logsumexp(log_alpha[t-1] + logA[:, j]) + logB[j, o[t]]
    return logsumexp(log_alpha[-1]), log_alpha

logp, log_alpha = forward_log(obs, pi, A, B)
print(f'前向 log P(O) = {logp:.6f}  ->  P(O) = {np.exp(logp):.6f}')
print(f'暴力        P(O) = {p_obs:.6f}')
assert np.allclose(np.exp(logp), p_obs, atol=1e-10), '前向必须对拍暴力枚举'
print('✅ 对数空间前向算法 == 暴力枚举 (到 1e-10) —— DP 精确, 非近似')

## 4 · Viterbi 解码（对拍暴力）

把前向的「求和」换成「取最大」并记回溯指针：$\delta_t(j)=\max_i[\delta_{t-1}(i)A_{ij}]B_{j,o_t}$。
暴力参考：枚举所有路径取联合概率最大的那条。

In [ ]:
def viterbi_log(obs, pi, A, B):
    o = [oi[x] for x in obs]; T = len(o)
    logpi, logA, logB = np.log(pi), np.log(A), np.log(B)
    delta = np.zeros((T, S)); psi = np.zeros((T, S), dtype=int)
    delta[0] = logpi + logB[:, o[0]]
    for t in range(1, T):
        for j in range(S):
            scores = delta[t-1] + logA[:, j]      # 各前驱状态的得分
            psi[t, j] = int(np.argmax(scores))    # 回溯指针: 最优前驱
            delta[t, j] = scores[psi[t, j]] + logB[j, o[t]]
    # 回溯
    best_last = int(np.argmax(delta[-1]))
    path = [best_last]
    for t in range(T - 1, 0, -1):
        path.append(psi[t, path[-1]])
    path.reverse()
    return [states[i] for i in path], float(delta[-1, best_last])

def brute_force_best(obs):
    best_path, best_lp = None, -np.inf
    for hidden in itertools.product(states, repeat=len(obs)):
        lp = np.log(joint_prob(obs, hidden))
        if lp > best_lp:
            best_lp, best_path = lp, list(hidden)
    return best_path, best_lp

vpath, vscore = viterbi_log(obs, pi, A, B)
bpath, bscore = brute_force_best(obs)
print('Viterbi 最优路径:', vpath, '  log-score =', round(vscore, 4))
print('暴力   最优路径:', bpath, '  log-score =', round(bscore, 4))
assert vpath == bpath, 'Viterbi 路径必须等于暴力最优'
assert np.allclose(vscore, bscore, atol=1e-10)
# 硬检验: Viterbi 得分 == 把这条路径代入联合概率
assert np.allclose(vscore, np.log(joint_prob(obs, vpath)), atol=1e-10)
print('✅ Viterbi == 暴力最优, 且得分==该路径联合概率 —— 解码正确')

## 5 · 后向算法与状态后验

$\beta_t(i)=\sum_j A_{ij}B_{j,o_{t+1}}\beta_{t+1}(j)$，$\beta_T=1$。
前向×后向给状态后验 $\gamma_t(i)=P(s_t=i|O)\propto\alpha_t(i)\beta_t(i)$，是 Baum-Welch 的 E 步。**健全性检验**：$\sum_i\alpha_t(i)\beta_t(i)=P(O)$ 对任意 $t$ 都成立。

In [ ]:
def backward_log(obs, pi, A, B):
    o = [oi[x] for x in obs]; T = len(o)
    logA, logB = np.log(A), np.log(B)
    log_beta = np.zeros((T, S))
    log_beta[-1] = 0.0                            # log 1 = 0
    for t in range(T - 2, -1, -1):
        for i in range(S):
            log_beta[t, i] = logsumexp(logA[i, :] + logB[:, o[t+1]] + log_beta[t+1])
    return log_beta

log_beta = backward_log(obs, pi, A, B)
# 对任意 t: logsumexp(log_alpha[t] + log_beta[t]) == log P(O)
for t in range(len(obs)):
    lp_t = logsumexp(log_alpha[t] + log_beta[t])
    assert np.allclose(lp_t, logp, atol=1e-10), f't={t} 前向后向不一致'
print('✅ 对任意 t, Σ_i α_t(i)β_t(i) = P(O) —— 前向后向一致')

# 状态后验 γ_t(i)
log_gamma = log_alpha + log_beta - logp
gamma = np.exp(log_gamma)
assert np.allclose(gamma.sum(1), 1.0, atol=1e-10), '每个位置的后验应归一'
print('状态后验 γ (每行和=1):')
for t, x in enumerate(obs):
    print(f'  t={t} ({x:5s}): P(Rainy)={gamma[t,0]:.3f} P(Sunny)={gamma[t,1]:.3f}')

## 6 · 监督参数估计（数频率 + 平滑）

有标签时，$\pi,A,B$ 全是计数比。从一个带标签的玩具语料估计参数，并验证每行归一。

In [ ]:
def estimate_supervised(tagged_sents, states, obs_vocab, smooth=0.1):
    '''tagged_sents: [[(word, tag), ...], ...]。返回 pi, A, B。'''
    S, Vo = len(states), len(obs_vocab)
    si = {s: i for i, s in enumerate(states)}
    oi = {o: i for i, o in enumerate(obs_vocab)}
    pi_c = np.zeros(S) + smooth
    A_c = np.zeros((S, S)) + smooth
    B_c = np.zeros((S, Vo)) + smooth
    for sent in tagged_sents:
        tags = [t for _, t in sent]
        pi_c[si[tags[0]]] += 1
        for (w, t) in sent:
            B_c[si[t], oi[w]] += 1
        for a, b in zip(tags[:-1], tags[1:]):
            A_c[si[a], si[b]] += 1
    pi = pi_c / pi_c.sum()
    A = A_c / A_c.sum(1, keepdims=True)
    B = B_c / B_c.sum(1, keepdims=True)
    return pi, A, B

# 玩具 POS 语料: N=名词, V=动词, D=限定词
tagged = [
    [('the','D'),('dog','N'),('runs','V')],
    [('the','D'),('cat','N'),('runs','V')],
    [('a','D'),('dog','N'),('barks','V')],
    [('the','D'),('cat','N'),('sleeps','V')],
]
pos_states = ['D', 'N', 'V']
pos_vocab = sorted({w for s in tagged for w, _ in s})
pi2, A2, B2 = estimate_supervised(tagged, pos_states, pos_vocab)
assert np.allclose(pi2.sum(), 1) and np.allclose(A2.sum(1), 1) and np.allclose(B2.sum(1), 1)
# 学到的结构: D 后面几乎总是 N
print('P(N | D) =', round(A2[0, 1], 3), '(限定词后面大概率是名词)')
assert A2[0, 1] > A2[0, 0] and A2[0, 1] > A2[0, 2]
print('✅ 监督估计: π,A,B 归一, 且学到 D->N 的语法规律')

---
## ✏️ 练习 1：前向算法（对数空间）

实现 `forward_logprob(obs, pi, A, B)`：在对数空间算 $\log P(O)$，返回标量。（用给定的 `logsumexp`、`oi`、`S`。）

In [ ]:
def forward_logprob(obs, pi, A, B):
    # TODO: log_alpha[0]=log pi + log B[:,o0]; 递推 log_alpha[t,j]=logsumexp(log_alpha[t-1]+logA[:,j])+logB[j,ot]
    #       返回 logsumexp(log_alpha[-1])
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
lp = forward_logprob(obs, pi, A, B)
assert np.allclose(np.exp(lp), brute_force_likelihood(obs), atol=1e-10), '应对拍暴力'
# 另一条观测也要对
obs2 = ['clean', 'clean', 'walk']
assert np.allclose(np.exp(forward_logprob(obs2, pi, A, B)),
                   brute_force_likelihood(obs2), atol=1e-10)
print('✅ 练习 1 通过：对数空间前向 == 暴力枚举')

## ✏️ 练习 2：Viterbi 解码

实现 `viterbi_decode(obs, pi, A, B)`：返回 (最优状态名列表, 最优 log-score)。对数空间，记回溯指针。

In [ ]:
def viterbi_decode(obs, pi, A, B):
    # TODO: delta[t,j]=max_i(delta[t-1,i]+logA[i,j])+logB[j,ot]; psi 记 argmax; 回溯
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
path, score = viterbi_decode(obs, pi, A, B)
bp, bs = brute_force_best(obs)
assert path == bp, f'Viterbi 路径应为暴力最优 {bp}, 得 {path}'
assert np.allclose(score, bs, atol=1e-10)
# 得分必须等于该路径的联合概率
assert np.allclose(score, np.log(joint_prob(obs, path)), atol=1e-10)
print('✅ 练习 2 通过：Viterbi == 暴力最优, 得分==路径联合概率')

## ✏️ 练习 3：后向算法

实现 `backward_log(obs, pi, A, B)`：返回 (T,S) 的 `log_beta`。$\beta_T=1$（log=0），$\beta_t(i)=\sum_j A_{ij}B_{j,o_{t+1}}\beta_{t+1}(j)$。正确性：对任意 $t$，$\text{logsumexp}(\log\alpha_t+\log\beta_t)=\log P(O)$。

In [ ]:
def backward_log(obs, pi, A, B):
    # TODO: log_beta[-1]=0; 倒推 log_beta[t,i]=logsumexp(logA[i,:]+logB[:,o_{t+1}]+log_beta[t+1])
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
lb = backward_log(obs, pi, A, B)
assert lb.shape == (len(obs), S)
assert np.allclose(lb[-1], 0.0), 'beta_T = 1 -> log=0'
la_ref = forward_log(obs, pi, A, B)[1]
lp_ref = forward_log(obs, pi, A, B)[0]
for t in range(len(obs)):
    assert np.allclose(logsumexp(la_ref[t] + lb[t]), lp_ref, atol=1e-10), f't={t} 不一致'
print('✅ 练习 3 通过：后向算法与前向一致 (任意 t 处 α·β 求和=P(O))')

## ✏️ 练习 4：监督参数估计

实现 `estimate_transition(tagged_sents, states)`：只返回转移矩阵 $A$（加 `smooth` 平滑、行归一）。$A_{ij}=\frac{\#(i\to j)}{\#(i\to\cdot)}$。

In [ ]:
def estimate_transition(tagged_sents, states, smooth=0.1):
    # TODO: 数相邻标签对 (i->j) 的频次(+smooth), 行归一
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
A_ex = estimate_transition(tagged, pos_states)
assert A_ex.shape == (3, 3)
assert np.allclose(A_ex.sum(1), 1.0), '每行应归一'
# D(0)->N(1) 应是 D 行里最大的
assert A_ex[0].argmax() == 1, 'D 后面最可能是 N'
# N(1)->V(2) 应是 N 行里最大的
assert A_ex[1].argmax() == 2, 'N 后面最可能是 V'
print('✅ 练习 4 通过：转移矩阵估计正确, 学到 D->N->V 的语法')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def forward_logprob(obs, pi, A, B):
    o = [oi[x] for x in obs]; T = len(o)
    logpi, logA, logB = np.log(pi), np.log(A), np.log(B)
    log_alpha = np.zeros((T, S))
    log_alpha[0] = logpi + logB[:, o[0]]
    for t in range(1, T):
        for j in range(S):
            log_alpha[t, j] = logsumexp(log_alpha[t-1] + logA[:, j]) + logB[j, o[t]]
    return logsumexp(log_alpha[-1])

In [ ]:
# 练习 2 参考答案
def viterbi_decode(obs, pi, A, B):
    o = [oi[x] for x in obs]; T = len(o)
    logpi, logA, logB = np.log(pi), np.log(A), np.log(B)
    delta = np.zeros((T, S)); psi = np.zeros((T, S), dtype=int)
    delta[0] = logpi + logB[:, o[0]]
    for t in range(1, T):
        for j in range(S):
            sc = delta[t-1] + logA[:, j]
            psi[t, j] = int(np.argmax(sc))
            delta[t, j] = sc[psi[t, j]] + logB[j, o[t]]
    last = int(np.argmax(delta[-1])); path = [last]
    for t in range(T - 1, 0, -1):
        path.append(psi[t, path[-1]])
    path.reverse()
    return [states[i] for i in path], float(delta[-1, last])

In [ ]:
# 练习 3 参考答案
def backward_log(obs, pi, A, B):
    o = [oi[x] for x in obs]; T = len(o)
    logA, logB = np.log(A), np.log(B)
    log_beta = np.zeros((T, S))
    for t in range(T - 2, -1, -1):
        for i in range(S):
            log_beta[t, i] = logsumexp(logA[i, :] + logB[:, o[t+1]] + log_beta[t+1])
    return log_beta

In [ ]:
# 练习 4 参考答案
def estimate_transition(tagged_sents, states, smooth=0.1):
    S = len(states); si = {s: i for i, s in enumerate(states)}
    A_c = np.zeros((S, S)) + smooth
    for sent in tagged_sents:
        tags = [t for _, t in sent]
        for a, b in zip(tags[:-1], tags[1:]):
            A_c[si[a], si[b]] += 1
    return A_c / A_c.sum(1, keepdims=True)

---
## 🧪 真实数据胶囊：用真实风格语料做 POS 标注

用真实风格的带词性标注语料（**尝试构造自 CoNLL 风格，失败回退内置真实标注句**），训练监督 HMM，用 Viterbi 给新句标注，算标注准确率。

In [ ]:
def load_pos_data():
    '''返回带标注句子 [[(word,tag),...],...]。内置一批真实英文 POS 标注句(通用词性)。'''
    # 真实英文句子 + 通用词性标注(D=det, N=noun, V=verb, P=prep, A=adj)
    data = [
        [('the','D'),('quick','A'),('fox','N'),('jumps','V')],
        [('a','D'),('lazy','A'),('dog','N'),('sleeps','V')],
        [('the','D'),('cat','N'),('sat','V'),('on','P'),('the','D'),('mat','N')],
        [('the','D'),('dog','N'),('ran','V'),('on','P'),('the','D'),('road','N')],
        [('a','D'),('big','A'),('bird','N'),('flew','V')],
        [('the','D'),('small','A'),('child','N'),('plays','V')],
        [('a','D'),('red','A'),('car','N'),('stops','V'),('on','P'),('a','D'),('hill','N')],
        [('the','D'),('old','A'),('man','N'),('walks','V')],
    ]
    return data, 'builtin real POS-tagged sentences'

data, src = load_pos_data()
print('数据来源:', src, '| 句数:', len(data))
train_pos, test_pos = data[:6], data[6:]
cap_states = sorted({t for s in data for _, t in s})    # 胶囊专用(别名, 避免覆盖前面的变量)
cap_vocab = sorted({w for s in data for w, _ in s})
print('词性:', cap_states, '| 词表大小:', len(cap_vocab))
assert len(train_pos) > 0 and len(test_pos) > 0
print('✅ 真实风格 POS 语料就绪')

**🧪 胶囊练习**：用 `estimate_supervised` 训练，对测试句用 `viterbi_log` 解码，统计标注准确率。补全训练与解码。

In [ ]:
# TODO: pi_p, A_p, B_p = estimate_supervised(train_pos, pos_states, pos_vocab)
#       对每个测试句, 用 viterbi 解码(注意: 需用 pos_states/pos_vocab 的索引版 viterbi)
#       这里直接用下面给出的 viterbi_generic 解码并统计 correct/total
raise NotImplementedError

In [ ]:
# 自测
assert 0.0 <= acc <= 1.0
assert acc >= 0.5, '监督 HMM 在这种规整语料上准确率应不低'
print(f'✅ 胶囊练习通过：POS 标注准确率 = {acc:.0%}')

In [ ]:
# 📖 胶囊参考答案
def viterbi_generic(obs, pi, A, B, states, oi_map):
    o = [oi_map[x] for x in obs]; T = len(o); Sn = len(states)
    logpi, logA, logB = np.log(pi), np.log(A), np.log(B)
    delta = np.zeros((T, Sn)); psi = np.zeros((T, Sn), dtype=int)
    delta[0] = logpi + logB[:, o[0]]
    for t in range(1, T):
        for j in range(Sn):
            sc = delta[t-1] + logA[:, j]
            psi[t, j] = int(np.argmax(sc)); delta[t, j] = sc[psi[t, j]] + logB[j, o[t]]
    last = int(np.argmax(delta[-1])); path = [last]
    for t in range(T - 1, 0, -1):
        path.append(psi[t, path[-1]])
    path.reverse(); return [states[i] for i in path]

pi_p, A_p, B_p = estimate_supervised(train_pos, cap_states, cap_vocab)
oi_p = {o: i for i, o in enumerate(cap_vocab)}
correct = total = 0
for sent in test_pos:
    words = [w for w, _ in sent]; gold = [t for _, t in sent]
    pred = viterbi_generic(words, pi_p, A_p, B_p, cap_states, oi_p)
    correct += sum(p == g for p, g in zip(pred, gold)); total += len(gold)
    print(' ', words, '->', pred, '(gold:', gold, ')')
acc = correct / total
print('标注准确率 =', round(acc, 3))

### 小结
- **HMM** = 生成式序列模型：隐状态按 $A$ 转移、按 $B$ 发射观测，参数 $\pi,A,B$。
- **三大问题**: 评估(前向)、解码(Viterbi)、学习(Baum-Welch/监督)，共用**动态规划**骨架。
- **前向** $\alpha_t(j)=(\sum_i\alpha_{t-1}(i)A_{ij})B_{j,o_t}$ 求和算 $P(O)$；**Viterbi** 把求和换 max + 回溯算最优路径。
- **对数空间 + log-sum-exp** 防下溢，是序列算法的生存技能。
- 我们对拍**暴力枚举**确认前向/Viterbi 精确(到 1e-10), 非近似。
- **监督估计**数频率最简单常用；**Baum-Welch**(EM)无监督但只到局部最优。

下一站：**模块 04 · CRF** —— 把 HMM 升级成判别式 + 任意特征, 但 DP 骨架(前向/Viterbi)原样保留。